In [ ]:
%pip install opencv-python
#Run if you don't have opencv installed.

In [43]:
import cv2
import numpy as np

In [44]:
# can change based on what color we want
COLOR = [
    (np.array([0,   120,  70]),  np.array([10,  255, 255])),  # lower red
    (np.array([170, 120,  70]),  np.array([180, 255, 255])),  # upper red
]

MIN_AREA = 800  # remove small items      
BLUR = (11, 11)  # blur

In [45]:
def mask(hsv: np.ndarray) -> np.ndarray:
    lower = cv2.inRange(hsv, COLOR[0][0], COLOR[0][1])
    upper = cv2.inRange(hsv, COLOR[1][0], COLOR[1][1])
    mask  = cv2.bitwise_or(lower, upper)
 
    # clean up mask
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return mask

In [46]:
def draw(frame: np.ndarray, mask: np.ndarray) -> np.ndarray:
    output = frame.copy()
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    found = 0

    for contour in contours:
        area = cv2.contourArea(contour)
        if area < MIN_AREA: continue
        found += 1
 
        x, y, w, h = cv2.boundingRect(contour)
        cv2.rectangle(output, (x, y), (x + w, y + h), (0, 0, 255), 2)
    return output

In [47]:
def detect_webcam(cam_index: int = 0):
    cap = cv2.VideoCapture(cam_index)
    if not cap.isOpened(): raise RuntimeError("no cam")
 
    while True:
        r, frame = cap.read()
        if not r: break
 
        hsv = cv2.cvtColor(cv2.GaussianBlur(frame, BLUR, 0), cv2.COLOR_BGR2HSV)
        m = mask(hsv)
        output = draw(frame, m)
 
        cv2.imshow("Detection", output)
        cv2.imshow("Mask", m)
 
        if cv2.waitKey(1) & 0xFF == ord("q"): break
 
    cap.release()
    cv2.destroyAllWindows()

In [49]:
detect_webcam()
#To change your camera, change the detect_wecam() 
#Parameter to your camera index (ex. dectect_webcam(1) for second camera)